# Data Manipulation with Pandas

Topics:
- GroupBy: split → apply → combine
- Merging and joining DataFrames
- Concatenating DataFrames
- Pivot tables and crosstabs
- Reshaping: melt, pivot, stack/unstack
- Rolling and expanding windows
- Handling duplicates

In [1]:
import pandas as pd
import numpy as np

# Sample sales dataset
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'date':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'region':   np.random.choice(['North','South','East','West'], n),
    'product':  np.random.choice(['Laptop','Phone','Tablet','Watch'], n),
    'rep':      np.random.choice(['Alice','Bob','Carol','Dave'], n),
    'units':    np.random.randint(1, 50, n),
    'price':    np.random.choice([999, 499, 299, 199], n),
})
df['revenue'] = df['units'] * df['price']
df['month'] = df['date'].dt.month_name()
print(df.head())
print(df.shape)

        date region product    rep  units  price  revenue    month
0 2024-01-01   East  Tablet   Dave     45    499    22455  January
1 2024-01-02   West   Watch   Dave     32    999    31968  January
2 2024-01-03  North  Tablet  Carol     30    199     5970  January
3 2024-01-04   East  Laptop  Alice     47    499    23453  January
4 2024-01-05   East   Watch   Dave     35    999    34965  January
(200, 8)


## 1. GroupBy — Split → Apply → Combine

The most powerful operation in Pandas. Group rows sharing a value, apply a function, combine results.

In [2]:
# Basic groupby + single aggregation
print(df.groupby('region')['revenue'].sum())
print()
print(df.groupby('region')['revenue'].mean().round(2))

region
East     856172
North    680682
South    581874
West     736205
Name: revenue, dtype: int64

region
East     15855.04
North    14797.43
South    12649.43
West     13633.43
Name: revenue, dtype: float64


In [3]:
# Multiple aggregations with .agg()
region_stats = df.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_units=('units', 'sum'),
    num_sales=('revenue', 'count')
).round(2)
print(region_stats)

        total_revenue  avg_revenue  total_units  num_sales
region                                                    
East           856172     15855.04         1528         54
North          680682     14797.43         1118         46
South          581874     12649.43         1226         46
West           736205     13633.43         1295         54


In [4]:
# Multi-level groupby
product_region = df.groupby(['product', 'region'])['revenue'].sum().unstack()
print(product_region.round(0))

region     East   North   South    West
product                                
Laptop   184226  178960  126578  219745
Phone    170348   90147  132679  117261
Tablet   249088  102725  135471  210970
Watch    252510  308850  187146  188229


In [5]:
# GroupBy + transform — keeps original index (great for adding group stats)
df['region_avg_revenue'] = df.groupby('region')['revenue'].transform('mean')
df['above_region_avg'] = df['revenue'] > df['region_avg_revenue']
print(df[['region','revenue','region_avg_revenue','above_region_avg']].head(10))

  region  revenue  region_avg_revenue  above_region_avg
0   East    22455        15855.037037              True
1   West    31968        13633.425926              True
2  North     5970        14797.434783             False
3   East    23453        15855.037037              True
4   East    34965        15855.037037              True
5   West    19960        13633.425926              True
6  North     4784        14797.434783             False
7  North     6487        14797.434783             False
8   East    20958        15855.037037              True
9  South    29970        12649.434783              True


In [6]:
# GroupBy + filter — keep only groups meeting a condition
big_regions = df.groupby('region').filter(lambda g: g['revenue'].sum() > 500000)
print('Regions with >500k revenue:')
print(big_regions['region'].unique())

Regions with >500k revenue:
<ArrowStringArray>
['East', 'West', 'North', 'South']
Length: 4, dtype: str


## 2. Merging & Joining DataFrames

Like SQL joins. `merge()` is the primary tool.

| Type | SQL | What it keeps |
|------|-----|----------------|
| `inner` | INNER JOIN | Only matching rows |
| `left` | LEFT JOIN | All left rows + matching right |
| `right` | RIGHT JOIN | All right rows + matching left |
| `outer` | FULL OUTER JOIN | All rows from both |

In [7]:
# Two tables to join
orders = pd.DataFrame({
    'order_id':   [1, 2, 3, 4, 5],
    'customer_id':[101, 102, 103, 101, 104],
    'amount':     [250, 150, 300, 175, 225]
})

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 105],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave'],
    'city':        ['NYC', 'LA', 'Chicago', 'Boston']
})

print('INNER join (only matched):')
print(pd.merge(orders, customers, on='customer_id', how='inner'))
print()
print('LEFT join (all orders):')
print(pd.merge(orders, customers, on='customer_id', how='left'))

INNER join (only matched):
   order_id  customer_id  amount   name     city
0         1          101     250  Alice      NYC
1         2          102     150    Bob       LA
2         3          103     300  Carol  Chicago
3         4          101     175  Alice      NYC

LEFT join (all orders):
   order_id  customer_id  amount   name     city
0         1          101     250  Alice      NYC
1         2          102     150    Bob       LA
2         3          103     300  Carol  Chicago
3         4          101     175  Alice      NYC
4         5          104     225    NaN      NaN


In [8]:
# Merge on different column names
orders2 = orders.rename(columns={'customer_id': 'cust_id'})
print(pd.merge(orders2, customers, left_on='cust_id', right_on='customer_id'))

# Merge on index
customers_idx = customers.set_index('customer_id')
print(orders.merge(customers_idx, left_on='customer_id', right_index=True))

   order_id  cust_id  amount  customer_id   name     city
0         1      101     250          101  Alice      NYC
1         2      102     150          102    Bob       LA
2         3      103     300          103  Carol  Chicago
3         4      101     175          101  Alice      NYC
   order_id  customer_id  amount   name     city
0         1          101     250  Alice      NYC
1         2          102     150    Bob       LA
2         3          103     300  Carol  Chicago
3         4          101     175  Alice      NYC


## 2b. From-Scratch — Manual Merge/Join

`pd.merge` looks like a black box, but an inner join is exactly this: build an index from the
right table's key to its matching rows, then for every left row emit one output row per match
found in that index. Implemented by hand with plain Python `dict`s on two toy tables, then
compared to `pd.merge`'s output on the same data.

In [9]:
left_table = {
    'id':   [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Carol', 'Dave'],
}
right_table = {
    'id':   [2, 3, 3, 5],
    'dept': ['Eng', 'Mkt', 'Eng', 'HR'],
}

# Build an index: right-table id -> list of matching row-dicts (the "hash join" build step)
right_index = {}
for rid, dept in zip(right_table['id'], right_table['dept']):
    right_index.setdefault(rid, []).append({'id': rid, 'dept': dept})

# INNER JOIN by hand: for every left row, emit one output row per match found in the index
manual_rows = []
for lid, name in zip(left_table['id'], left_table['name']):
    for match in right_index.get(lid, []):
        manual_rows.append({'id': lid, 'name': name, 'dept': match['dept']})

manual_df = pd.DataFrame(manual_rows)
print('Manual inner join:')
print(manual_df)

# Same two tables, pd.merge
left_df = pd.DataFrame(left_table)
right_df = pd.DataFrame(right_table)
pandas_df = pd.merge(left_df, right_df, on='id', how='inner')
print('\npd.merge inner join:')
print(pandas_df)

matches = manual_df.reset_index(drop=True).equals(pandas_df.reset_index(drop=True))
print('\nManual join matches pd.merge exactly:', matches)

Manual inner join:
   id   name dept
0   2    Bob  Eng
1   3  Carol  Mkt
2   3  Carol  Eng

pd.merge inner join:
   id   name dept
0   2    Bob  Eng
1   3  Carol  Mkt
2   3  Carol  Eng

Manual join matches pd.merge exactly: True


## 3. Concatenating DataFrames

In [10]:
q1 = pd.DataFrame({'month':['Jan','Feb','Mar'], 'sales':[100,120,110]})
q2 = pd.DataFrame({'month':['Apr','May','Jun'], 'sales':[130,125,140]})
q3 = pd.DataFrame({'month':['Jul','Aug','Sep'], 'sales':[150,145,160]})

# Vertical stack (axis=0 — default)
year = pd.concat([q1, q2, q3], ignore_index=True)
print(year)

# Horizontal stack (axis=1) — columns side by side
left  = pd.DataFrame({'name': ['Alice','Bob'], 'age': [25, 30]})
right = pd.DataFrame({'score': [88, 92],       'grade': ['B','A']})
print(pd.concat([left, right], axis=1))

  month  sales
0   Jan    100
1   Feb    120
2   Mar    110
3   Apr    130
4   May    125
5   Jun    140
6   Jul    150
7   Aug    145
8   Sep    160
    name  age  score grade
0  Alice   25     88     B
1    Bob   30     92     A


## 4. Pivot Tables

Pivot tables summarise data by two dimensions — like an Excel pivot table.

In [11]:
# Revenue by product × region
pivot = df.pivot_table(
    values='revenue',
    index='product',
    columns='region',
    aggfunc='sum',
    fill_value=0,
    margins=True,    # add 'All' row and column totals
    margins_name='Total'
)
print(pivot.round(0))

# Crosstab — frequency counts
print(pd.crosstab(df['product'], df['region']))

region     East   North   South    West    Total
product                                         
Laptop   184226  178960  126578  219745   709509
Phone    170348   90147  132679  117261   510435
Tablet   249088  102725  135471  210970   698254
Watch    252510  308850  187146  188229   936735
Total    856172  680682  581874  736205  2854933
region   East  North  South  West
product                          
Laptop     13     12     11    13
Phone      12      7      9    12
Tablet     17      9     10    14
Watch      12     18     16    15


## 5. Reshaping: melt, pivot, stack/unstack

In [12]:
# Wide format (one row per student, one col per subject)
wide = pd.DataFrame({
    'student': ['Alice','Bob','Carol'],
    'math':    [85, 92, 78],
    'english': [90, 88, 95],
    'science': [82, 79, 91]
})
print('Wide:')
print(wide)

# melt → long format (better for groupby and plotting)
long = wide.melt(id_vars='student', var_name='subject', value_name='score')
print('\nLong:')
print(long)

# pivot → back to wide from long
back_to_wide = long.pivot(index='student', columns='subject', values='score')
print('\nBack to wide:')
print(back_to_wide)

Wide:
  student  math  english  science
0   Alice    85       90       82
1     Bob    92       88       79
2   Carol    78       95       91

Long:
  student  subject  score
0   Alice     math     85
1     Bob     math     92
2   Carol     math     78
3   Alice  english     90
4     Bob  english     88
5   Carol  english     95
6   Alice  science     82
7     Bob  science     79
8   Carol  science     91

Back to wide:
subject  english  math  science
student                        
Alice         90    85       82
Bob           88    92       79
Carol         95    78       91


In [13]:
# stack and unstack — work with MultiIndex
multi = df.groupby(['product','region'])['revenue'].sum()
print('MultiIndex Series:')
print(multi.head(8))
print('\nUnstacked (product × region):')
print(multi.unstack())
print('\nStacked back:')
print(multi.unstack().stack().head(8))

MultiIndex Series:
product  region
Laptop   East      184226
         North     178960
         South     126578
         West      219745
Phone    East      170348
         North      90147
         South     132679
         West      117261
Name: revenue, dtype: int64

Unstacked (product × region):
region     East   North   South    West
product                                
Laptop   184226  178960  126578  219745
Phone    170348   90147  132679  117261
Tablet   249088  102725  135471  210970
Watch    252510  308850  187146  188229

Stacked back:
product  region
Laptop   East      184226
         North     178960
         South     126578
         West      219745
Phone    East      170348
         North      90147
         South     132679
         West      117261
dtype: int64


## 6. Rolling & Expanding Windows

Used for time series smoothing and cumulative statistics.

In [14]:
# Daily sales aggregated
daily = df.groupby('date')['revenue'].sum().reset_index()
daily = daily.sort_values('date')

# 7-day rolling average
daily['rolling_7d_avg'] = daily['revenue'].rolling(window=7).mean()

# Cumulative sum
daily['cumulative_revenue'] = daily['revenue'].cumsum()

# Expanding mean (average of all data up to that point)
daily['expanding_avg'] = daily['revenue'].expanding().mean()

print(daily.head(15).to_string(index=False))

      date  revenue  rolling_7d_avg  cumulative_revenue  expanding_avg
2024-01-01    22455             NaN               22455   22455.000000
2024-01-02    31968             NaN               54423   27211.500000
2024-01-03     5970             NaN               60393   20131.000000
2024-01-04    23453             NaN               83846   20961.500000
2024-01-05    34965             NaN              118811   23762.200000
2024-01-06    19960             NaN              138771   23128.500000
2024-01-07     4784    20507.857143              143555   20507.857143
2024-01-08     6487    18226.714286              150042   18755.250000
2024-01-09    20958    16653.857143              171000   19000.000000
2024-01-10    29970    20082.428571              200970   20097.000000
2024-01-11    18981    19443.571429              219951   19995.545455
2024-01-12    16983    16874.714286              236934   19744.500000
2024-01-13     9481    15377.714286              246415   18955.000000
2024-0

## 7. Handling Duplicates

In [15]:
dupes = pd.DataFrame({
    'name':  ['Alice','Bob','Alice','Carol','Bob','Alice'],
    'score': [85, 92, 85, 78, 92, 90]
})

print('Duplicates:')
print(dupes.duplicated())              # True for duplicate rows
print('Count:', dupes.duplicated().sum())

print('\nDuplicate rows:')
print(dupes[dupes.duplicated(keep=False)])   # show all copies

print('\nDrop exact duplicates:')
print(dupes.drop_duplicates())

print('\nKeep first by name only:')
print(dupes.drop_duplicates(subset=['name'], keep='first'))

Duplicates:
0    False
1    False
2     True
3    False
4     True
5    False
dtype: bool
Count: 2

Duplicate rows:
    name  score
0  Alice     85
1    Bob     92
2  Alice     85
4    Bob     92

Drop exact duplicates:
    name  score
0  Alice     85
1    Bob     92
3  Carol     78
5  Alice     90

Keep first by name only:
    name  score
0  Alice     85
1    Bob     92
3  Carol     78


## 8. Failure Modes

- **Row-count explosion in a many-to-many join.** If both tables have repeated keys, `merge`
  produces the *Cartesian product* of matching rows for that key, not one row per input row.
- **Inner join silently drops unmatched rows.** No error, no warning — rows on either side with
  no match on the other simply disappear from the output.

In [16]:
print('left rows:', len(left_df), '  right rows:', len(right_df))
print('inner join rows:', len(pandas_df))
print('left ids dropped by the inner join:', set(left_df['id']) - set(pandas_df['id']))
print('right ids dropped by the inner join:', set(right_df['id']) - set(pandas_df['id']))

# Many-to-many join: repeated keys on BOTH sides -> Cartesian product per key
m2m_left  = pd.DataFrame({'key': ['a', 'a', 'b'], 'lval': [1, 2, 3]})
m2m_right = pd.DataFrame({'key': ['a', 'a', 'b'], 'rval': [10, 20, 30]})
exploded = pd.merge(m2m_left, m2m_right, on='key')
print(f"\nleft rows: {len(m2m_left)}, right rows: {len(m2m_right)} -> merged rows: {len(exploded)}")
print(exploded)

left rows: 4   right rows: 4
inner join rows: 3
left ids dropped by the inner join: {1, 4}
right ids dropped by the inner join: {5}

left rows: 3, right rows: 3 -> merged rows: 5
  key  lval  rval
0   a     1    10
1   a     1    20
2   a     2    10
3   a     2    20
4   b     3    30


## Quick Summary

| Operation | Key Function |
|-----------|--------------|
| Group & aggregate | `df.groupby('col').agg(...)` |
| Add group stats | `.groupby().transform()` |
| SQL-style join | `pd.merge(left, right, on='key', how='inner')` |
| Stack vertically | `pd.concat([df1,df2], ignore_index=True)` |
| Summarise 2 dims | `df.pivot_table(values, index, columns, aggfunc)` |
| Wide → Long | `df.melt(id_vars=..., var_name=..., value_name=...)` |
| Long → Wide | `df.pivot(index, columns, values)` |
| Moving average | `.rolling(7).mean()` |
| Running total | `.cumsum()` |
| Find duplicates | `.duplicated()`, `.drop_duplicates()` |

**Next →** [04 – Reading Data](../04-data-reading/)